## D2 - Two-Model Decision Making: Wall, Red, and Blue
Author: George Gorospe, george.gorospe@nmaia.net\
Last Update: July 22, 2026

### About: This morning your robot learned to stop and turn when something was in the way -- but it couldn't tell *what* it was avoiding. Today you'll combine your Free/Blocked model with the new Wall/Red/Blue model from D1, so your robot can make a smarter decision: wall, red, or blue each get a different response.

### Today's Bigger Challenge: Using Two Models Together
### You're not building anything from scratch here -- you're combining two things you already have: the Free/Blocked model from this morning, and the Wall/Red/Blue model you just started training in D1.

### Decision Spec
### - **Free** -- keep driving forward.
### - **Blocked**, and the obstacle is a **wall** -- turn 180 degrees (relative) and continue.
### - **Blocked**, and the obstacle is **red** -- turn left.
### - **Blocked**, and the obstacle is **blue** -- turn right.

### Notice the two-step shape of this decision: first ask *is the path blocked at all*, and only if it is, ask *what's blocking it*. That's exactly the two-model pipeline we benchmarked earlier this week -- the second model only runs when it's actually needed.

### Why Flowchart First -- Especially Now
### This morning's flowcharts had one question and two branches. Today's has one question that leads to a *second* question with three more branches underneath it. More branches means more value in planning before coding -- it's much easier to spot a missing case on paper than while debugging a moving robot.

### 🤖 Real Robotics Engineering: Sensor Fusion
### Combining the outputs of multiple models (or sensors) to make one decision is literally called **sensor fusion** in real robotics systems. You're about to do, in miniature, what a self-driving car does when it combines camera and radar data to make a single driving decision.

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY D2.1: Designing Your Flowchart</span>

### On paper (or a whiteboard), sketch the full flowchart for today's behavior before writing any code. A few questions to work through as you draw it:
### - What does the robot sense first, and what are the possible answers?
### - For each answer, what happens next -- is there a second question, or a direct action?
### - After a turn (any of the three), what should the robot do? Keep driving forward, and let the next sensing cycle figure out if there's a new obstacle? Or something else?
### - Does every branch eventually lead back to "keep driving" so the robot doesn't get stuck?

### **This part doesn't need the robot or this notebook.** If your Wall/Red/Blue model is still training from D1, this is exactly the gap that training time is meant to fill -- work through your flowchart with your team now, and come back to Part 2 below once you're ready (whether or not training has finished).

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY D2.2: Building the Two-Model Decision Program</span>

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 1. Import required libraries

from sphero_sdk import RawMotorModesEnum
import ipywidgets as widgets
from IPython.display import display
from ipyfilechooser import FileChooser

from jetcam_lite import TraitletCamera, bgr8_to_jpeg

from robot_utils import get_rvr, close_if_exists
from jupyter_utils import register_dlink
from inference_utils import load_model_and_metadata, show_inference_grid
from behavior_utils import start_two_stage_behavior_loop, create_start_stop_buttons, stop_behavior_loop

### STEP 2. Choose both models: your Free/Blocked model from this morning, and your Wall/Red/Blue model from D1.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

print("Choose your FREE/BLOCKED model:")
free_blocked_chooser = FileChooser('/home/explorer/Models/')
free_blocked_chooser.filter_pattern = '*.pth'
display(free_blocked_chooser)

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

print("Choose your WALL/RED/BLUE model:")
wall_red_blue_chooser = FileChooser('/home/explorer/Models/')
wall_red_blue_chooser.filter_pattern = '*.pth'
display(wall_red_blue_chooser)

### STEP 3. Load both models. You should see a training summary for each.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

free_blocked_model, free_blocked_device, free_blocked_classes, free_blocked_record = \
    load_model_and_metadata(free_blocked_chooser.selected)

print()

wall_red_blue_model, wall_red_blue_device, wall_red_blue_classes, wall_red_blue_record = \
    load_model_and_metadata(wall_red_blue_chooser.selected)

### Quick accuracy check on your new Wall/Red/Blue model -- 8 random images from its dataset, true label vs. predicted label.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

show_inference_grid(
    wall_red_blue_model,
    wall_red_blue_classes,
    wall_red_blue_device,
    wall_red_blue_record['dataset_dir']
)

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 4. Connect to the robot and start the camera.

rvr = get_rvr()

# Reset yaw so heading 0 has a known, predictable meaning -- this notebook
# tracks the robot's current heading itself (see heading_state below), and
# that tracking is only meaningful if it starts from a known reference.
rvr.reset_yaw()

camera = TraitletCamera()
camera.start()

image_widget = widgets.Image(format='jpeg', width=camera.width, height=camera.height)
register_dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)
display(image_widget)

### STEP 5. Now write the decision logic into the ```decide_action``` function:  (READ THIS PART CAREFULLY)
###  This function receives **two** labels:  
###  `primary_label` is always the Free/Blocked prediction; this happens first.  
###  `secondary_label` is the Wall/Red/Blue prediction, this happens only when `primary_label == 'blocked'` -- otherwise it's `None`.

### We use the labels as decision points: 
### **free** --> keep driving.  
### **blocked** --> move on to wall/red/blue model to decide what is in front of us and act accordingly.  

## --------------------------------------

### The `'free'` branch is already filled in for you: it's the same "keep driving" idea as C1/C2. We're just using `drive_with_heading()`.   
### Notice it also tracks `heading_state['current']` any time the robot turns, so "drive forward" always means "continue in whatever direction I'm currently facing," not always due north.

### Fill in the three blanks for wall, red, and blue. Each one needs to:
1. Update `heading_state['current']` to the robot's new heading after turning. Example: ```heading_state['current'] = 270``` for LEFT.
2. Call `rvr.drive_with_heading(speed, heading_state['current'], 0)` with that new heading.

### For wall specifically, remember: this is a **relative** turn (180 degrees from wherever the robot currently is), not a fixed compass direction -- so the new heading is `heading_state['current'] + 180`, not just `180`.

## --------------------------------------

### Why update `heading_state['current']` at all?
### Look again at the `'free'` branch: every time it runs, it drives using whatever value is *currently stored* in `heading_state['current']` -- not a fixed number. That's what lets `'free'` mean "keep going the way I'm currently facing," instead of "always drive due north."
### That only works if turning actually updates what's stored there. If you call `drive_with_heading()` with a new heading but never update `heading_state['current']` to match, the *robot* turns, but the notebook's *memory* of which way it's facing doesn't. The very next time the path is free, `decide_action` will drive using the old, now-incorrect heading -- silently undoing the turn you just made.
### In short: `heading_state['current']` isn't just bookkeeping. It's the one thing keeping the `'free'` branch honest about which way the robot is actually pointed, cycle to cycle.

In [ ]:
#### ------> ACTIVITY D2.2: decide_action function <-----#####
# About: this function is called repeatedly while the behavior is running.
# primary_label is the Free/Blocked prediction (always present).
# secondary_label is the Wall/Red/Blue prediction -- only present
# (not None) when primary_label == 'blocked'.

##### INSTRUCTIONS: #####
# 1. In the 'wall' branch, turn 180 degrees RELATIVE to the current heading.
# 2. In the 'red' branch, turn left.
# 3. In the 'blue' branch, turn right.
# In each case: update heading_state['current'], then call drive_with_heading().

heading_state = {'current': 0}

def decide_action(rvr, primary_label, secondary_label):
    if primary_label == 'free':
        # Already done for you -- keep driving in the current direction.
        rvr.drive_with_heading(100, heading_state['current'], 0)

    else:  # primary_label == 'blocked'
        if secondary_label == 'wall':
            #<<<<<< replace this with your code >>>>>>
            pass

        elif secondary_label == 'red':
            #<<<<<< replace this with your code >>>>>>
            pass

        else:  # secondary_label == 'blue'
            #<<<<<< replace this with your code >>>>>>
            pass

### STEP 6. Build the Start/Stop buttons. Running this cell does **not** move the robot -- nothing happens until you press Start.

######## WILL CAUSE ROBOT MOTION ONCE STARTED: ENSURE ROBOT IS ON THE GROUND, WITH SPACE TO TURN, AND WALL/RED/BLUE OBSTACLES NEARBY TO TEST AGAINST #########

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

buttons = create_start_stop_buttons(
    lambda: start_two_stage_behavior_loop(
        rvr, camera,
        free_blocked_model, free_blocked_classes, free_blocked_device,
        wall_red_blue_model, wall_red_blue_classes, wall_red_blue_device,
        decide_action,
        trigger_label='blocked'
    )
)
display(buttons)

### Press **Start**, then test all three obstacle types: a wall, a red block, and a blue block. Watch whether each one gets the response you designed in your flowchart. Press **Stop** any time to immediately halt the behavior.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 7. Stop the behavior.
stop_behavior_loop()

<span style="color: green; font-size: 55px; font-style: italic;">Student Discussion Time</span>

### Talk through these questions with your team:
### - Did each obstacle type get the response your flowchart planned? Any surprises?
### - Could you notice any delay on the "blocked" branch compared to "free," now that two models are running instead of one? (We measured this earlier this week -- does what you observed match what the benchmark predicted?)
### - After a turn, the robot goes right back to sensing on the very next loop cycle. What happens if it turns toward *another* obstacle? Does your flowchart -- and your code -- actually handle that, or does it assume one turn is always enough?
### - What would you need to add to handle a case your flowchart didn't originally plan for?

## Excellent work -- your robot is now making real decisions from two models working together, exactly like real robotic and autonomous vehicle systems do.

## **NEXT**: D3 reintroduces dead reckoning for a new "seeking" behavior -- driving toward and touching a blue block instead of just avoiding it.

# SOLUTIONS:

### ------> SOLUTION - Activity D2.2: decide_action function <-----

In [ ]:
#### ------> SOLUTION - Activity D2.2: decide_action function <-----#####
# About: this function is called repeatedly while the behavior is running.
# primary_label is the Free/Blocked prediction (always present).
# secondary_label is the Wall/Red/Blue prediction -- only present
# (not None) when primary_label == 'blocked'.

heading_state = {'current': 0}

def decide_action(rvr, primary_label, secondary_label):
    if primary_label == 'free':
        # Already done for you -- keep driving in the current direction.
        rvr.drive_with_heading(100, heading_state['current'], 0)

    else:  # primary_label == 'blocked'
        if secondary_label == 'wall':
            # GEORGE's Solution:
            # 180 degrees RELATIVE to the current heading -- add 180 to
            # whatever heading_state['current'] already holds, don't just
            # set it to 180.
            heading_state['current'] = (heading_state['current'] + 180) % 360
            rvr.drive_with_heading(100, heading_state['current'], 0)

        elif secondary_label == 'red':
            # GEORGE's Solution:
            heading_state['current'] = 270
            rvr.drive_with_heading(100, heading_state['current'], 0)

        else:  # secondary_label == 'blue'
            # GEORGE's Solution:
            heading_state['current'] = 90
            rvr.drive_with_heading(100, heading_state['current'], 0)

### Wrapping Up
Before you move on to the next notebook, run the cell below to release the robot's connection.

In [ ]:
#### ------> RUN THIS CELL WHEN YOU'RE DONE WITH THIS NOTEBOOK <-----#####
close_if_exists()
print("Robot connection closed. Safe to move on to the next notebook!")